In [6]:
import jax
import jax.numpy as jnp
import optax
import time
import qutip as qt

from qst_tec.basicfunc import custom_nuclear_norm, build_separable_state, distance_sq, partial_transpose, ppt_penalty, ppt_value

# 配置使用双精度 (建议放在所有 jax 操作的最前面)
jax.config.update("jax_enable_x64", True)


In [7]:

# ==========================================
# 第1部分：构造待验证的量子态 (紧凑化)
# ==========================================
d = 4
k0, k1, k2 = qt.basis(d, 0), qt.basis(d, 1), qt.basis(d, 2)

v0 = qt.tensor(k0, (k0 - k1).unit())
v1 = qt.tensor((k0 - k1).unit(), k2)
v2 = qt.tensor(k2, (k1 - k2).unit())
v3 = qt.tensor((k1 - k2).unit(), k0)
v4 = qt.tensor((k0 + k1 + k2).unit(), (k0 + k1 + k2).unit())

P_UPB = v0.proj() + v1.proj() + v2.proj() + v3.proj() + v4.proj()
I_9 = sum([qt.tensor(qt.basis(d, i), qt.basis(d, j)).proj() for i in range(3) for j in range(3)])

rho_bes_qutip = 0.98 * (I_9 - P_UPB) / 4.0 + 0.02 * qt.tensor(qt.qeye(4), qt.qeye(4)) / 16.0

rho_target = jnp.array(rho_bes_qutip.full(), dtype=jnp.complex128)


In [8]:
# ==========================================
# 第2部分：定义验证器 Loss
# ==========================================
def verifier_loss(params_V, rho_targ):
    rho_sep = build_separable_state(params_V)
    return distance_sq(rho_targ, rho_sep)


In [9]:

# ==========================================
# 第3部分：训练主体 (结合 Optax 调度 与 JAX Scan)
# ==========================================
def verifier_separable_state(rho_verified, d=4, K=None, steps=2000):
    if K is None:
        K = int(rho_verified.shape[0] ** 2)

    key = jax.random.PRNGKey(42)
    k1, k2, k3, k4 = jax.random.split(key, 4)

    A = jax.random.normal(k1, (K, d)) + 1j * jax.random.normal(k2, (K, d))
    B = jax.random.normal(k3, (K, d)) + 1j * jax.random.normal(k4, (K, d))
    params_V = (A, B)
    
    # 🌟 优化点 1：引入 Optax 动态学习率调度
    # 这里使用余弦退火，初始学习率 0.1，到末尾平滑衰减到 0.001
    schedule = optax.cosine_decay_schedule(init_value=0.1, decay_steps=steps, alpha=0.01)
    opt_V = optax.adam(learning_rate=schedule)
    opt_state_V = opt_V.init(params_V)

    # 🌟 优化点 2：定义专门供 jax.lax.scan 使用的单步更新函数
    # scan 要求函数的签名是 f(carry_state, x) -> (new_carry_state, y)
    def train_step(state, step_idx):
        params, opt_state = state
        loss_val, grads = jax.value_and_grad(verifier_loss)(params, rho_verified)
        
        # 规范化：使用 jax.tree_util.tree_map 处理复数共轭 (Wirtinger calculus)
        grads = jax.tree_util.tree_map(jnp.conj, grads)
        
        updates, new_opt_state = opt_V.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        
        # 如果你想在 XLA 编译期间实现定步长打印，可以使用 jax.debug.callback
        # 否则可以不写，直接返回 loss_val 以供最后绘图
        jax.debug.callback(
            lambda s, l, lr: print(f"Step {s:4d} | 距离: {l:.6f} | 学习率: {lr:.4f}") if s % 200 == 0 else None,
            step_idx, loss_val, schedule(step_idx)
        )
        
        return (new_params, new_opt_state), loss_val

    print(f"开始搜索可分态分解: d={d}, K={K}, 计划步数: {steps}")
    start_time = time.time()
    
    # 🌟 优化点 3：使用 jax.lax.scan 编译整个循环为一张图
    init_state = (params_V, opt_state_V)
    
    # final_state 包含训练结束后的 params 和 opt_state
    # loss_history 自动收集了所有 step 的 loss 记录，方便后续画图
    final_state, loss_history = jax.lax.scan(train_step, init_state, jnp.arange(steps))
    
    print(f"耗时: {time.time() - start_time:.4f}s")
    
    final_params = final_state[0]
    return final_params, loss_history


In [10]:

# ==========================================
# 第4部分：执行与验证
# ==========================================
final_V_params, losses = verifier_separable_state(rho_target, d=4, K=256, steps=2000)

print("\n=== 最终结果 ===")
final_rho_sep = build_separable_state(final_V_params)
print("最终态的与可分态的 Frobenius 范数平方距离:", distance_sq(final_rho_sep, rho_target))
print("最终态的 PPT value:", ppt_penalty(final_rho_sep))
print("目标态的 PPT value:", ppt_value(rho_target))

开始搜索可分态分解: d=4, K=256, 计划步数: 2000
Step    0 | 距离: 0.185879 | 学习率: 0.1000
Step  200 | 距离: 0.001410 | 学习率: 0.0976
Step  400 | 距离: 0.001384 | 学习率: 0.0905
Step  600 | 距离: 0.001380 | 学习率: 0.0796
Step  800 | 距离: 0.001379 | 学习率: 0.0658
Step 1000 | 距离: 0.001378 | 学习率: 0.0505
Step 1200 | 距离: 0.001378 | 学习率: 0.0352
Step 1400 | 距离: 0.001378 | 学习率: 0.0214
Step 1600 | 距离: 0.001378 | 学习率: 0.0105
Step 1800 | 距离: 0.001378 | 学习率: 0.0034
耗时: 0.8635s

=== 最终结果 ===
最终态的与可分态的 Frobenius 范数平方距离: 0.001377939759849167
最终态的 PPT value: 0.0
目标态的 PPT value: 0.0012499999999998346
